# 🐾 Animal Sound Generator — Colab Training v12

**Kill the Electric Sound.** Diffusion with spectral balance + temporal smoothness + classifier guidance.

| Step | Model | Time |
|------|-------|------|
| 1 | Diffusion UNet (~4M params) | ~2 hrs |
| 2 | Generate & listen | 5 sec |

### V12 Changes:
- Smaller model (4M vs 18M params) — prevents overfitting on 640 samples
- 7 animal classes (Noise removed)
- Spectral balance loss — forces realistic frequency distribution
- Temporal smoothness loss — kills frame-to-frame jitter
- Classifier guidance — ensures animal-like structure
- Spectral stats matching — fixes HiFi-GAN input

### Before running:
1. Runtime → L4 GPU
2. Push your code to GitHub first

In [ ]:
# @title 1. Setup
!git clone https://github.com/grindydev/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib pandas scikit-learn librosa soundfile tqdm

!mkdir -p models/diffusion_checkpoints/train

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Download ESC-50 Data (640 clean animal sound clips)
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip
!unzip -qo /tmp/esc50.zip -d /tmp/
!python src/scripts/setup_esc50.py --source /tmp/ESC-50-master/audio --target data/esc50

# Restore existing checkpoints from Drive (optional)
import os
DRIVE = "/content/drive/MyDrive/animal_sound_generator/models"
if os.path.isdir(DRIVE):
    !cp -r {DRIVE}/* models/ 2>/dev/null
    !ls -lh models/*.pth 2>/dev/null || echo "(no .pth files yet)"

!ls data/esc50/

In [ ]:
# @title ⭐ Train Diffusion v12 — ESC-50 (150 epochs)
# Smaller model (4M params), 7 animal classes, spectral balance + smoothness + classifier guidance
!python src/diffusion/train.py


In [ ]:
# @title 3. Generate Audio (DDIM 200 steps)
!python src/generate.py --label Dog --from-scratch --diffusion-steps 200

In [ ]:
# @title 💾 Save Checkpoints to Drive
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/animal_sound_generator/models"
!mkdir -p {DRIVE}
import os
for f in ["diffusion_unet_train_best.pth", "diffusion_unet_train.pth"]:
    path = f"models/{f}"
    if os.path.exists(path):
        !cp {path} {DRIVE}/
        print(f"  ✅ {f} ({os.path.getsize(path)/1e6:.0f} MB)")
!cp -r models/diffusion_checkpoints {DRIVE}/ 2>/dev/null
print(f"
📂 Saved to {DRIVE}/")

In [ ]:
# @title 🎧 Generate & Listen
!python src/generate.py --label Dog --from-scratch --diffusion-steps 200

from IPython.display import Audio, display
import glob
wavs = sorted(glob.glob("outputs/generated/*.wav"))
if wavs:
    for w in wavs[-3:]:
        display(Audio(w, rate=22050))
        print(f"🔊 {w}")
else:
    print("No audio — train diffusion first")